## Setup

In [1]:
import os
import sys
import time
import logging
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, str(Path("../").resolve()))

from src.part2.router import classify_query
from src.part2.retriever import retrieve_from_csv, retrieve_from_text
from src.part2.pipeline import build_pipeline, answer_question

load_dotenv(dotenv_path="../.env")

logging.basicConfig(
    format="[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

/Users/qing0304/Desktop/GU/DSAN6725/spring-2026-a03-Cynthia2734/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Verify API key is set
if os.getenv("GROQ_API_KEY"):
    logger.info("GROQ_API_KEY is set")
elif os.getenv("OPENAI_API_KEY"):
    logger.info("OPENAI_API_KEY is set")
elif os.getenv("ANTHROPIC_API_KEY"):
    logger.info("ANTHROPIC_API_KEY is set")
else:
    logger.warning("No API key found! Please set GROQ_API_KEY in your .env file")

# Check for Cohere API key (needed for re-ranking)
if os.getenv("COHERE_API_KEY"):
    logger.info("COHERE_API_KEY is set (for re-ranking)")
else:
    logger.warning("COHERE_API_KEY not set - re-ranking section will not work")

[2026-03-05 00:45:41,399] p41196 {21338686.py:3} INFO - GROQ_API_KEY is set
[2026-03-05 00:45:41,399] p41196 {21338686.py:13} INFO - COHERE_API_KEY is set (for re-ranking)


## Configuration

In [3]:
CSV_PATH = "../data/structured/daily_sales.csv"
TEXT_DIR = "../data/unstructured"
MODEL_ID = "groq/llama-3.1-8b-instant"

text_files = list(Path(TEXT_DIR).glob("*.txt")) if Path(TEXT_DIR).exists() else []
logger.info(f"Found {len(text_files)} product text files in {TEXT_DIR}")

# Build pipeline once, reuse for all questions
pipeline = build_pipeline(model_id=MODEL_ID, csv_path=CSV_PATH, text_dir=TEXT_DIR)

test_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?",
]

[2026-03-05 00:45:41,404] p41196 {563378281.py:6} INFO - Found 10 product text files in ../data/unstructured
/Users/qing0304/Desktop/GU/DSAN6725/spring-2026-a03-Cynthia2734/src/part2/pipeline.py:57: LangChainDeprecationWarning: The class `ChatLiteLLM` was deprecated in LangChain 0.3.24 and will be removed in 1.0. An updated version of the class exists in the `langchain-litellm package and should be used instead. To use it run `pip install -U `langchain-litellm` and import as `from `langchain_litellm import ChatLiteLLM``.
  self.llm = ChatLiteLLM(model=model_id, temperature=0)
[2026-03-05 00:45:42,272] p41196 {pipeline.py:58} INFO - Part2Pipeline initialised (model=groq/llama-3.1-8b-instant)


## Explore Data

In [4]:
df = pd.read_csv(CSV_PATH)
logger.info(f"CSV shape: {df.shape}")
logger.info(f"Columns: {list(df.columns)}")
logger.info(f"Categories: {df['category'].unique().tolist()}")
logger.info(f"Regions: {df['region'].unique().tolist()}")
logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head(3)

[2026-03-05 00:45:42,278] p41196 {2528254882.py:2} INFO - CSV shape: (1000, 8)
[2026-03-05 00:45:42,279] p41196 {2528254882.py:3} INFO - Columns: ['date', 'product_id', 'product_name', 'category', 'units_sold', 'unit_price', 'total_revenue', 'region']
[2026-03-05 00:45:42,280] p41196 {2528254882.py:4} INFO - Categories: ['Books', 'Electronics', 'Food & Grocery', 'Home & Kitchen', 'Pet Supplies', 'Office Supplies', 'Toys & Games', 'Sports & Outdoors', 'Beauty & Personal Care', 'Clothing']
[2026-03-05 00:45:42,280] p41196 {2528254882.py:5} INFO - Regions: ['Central', 'West', 'North', 'South', 'East']
[2026-03-05 00:45:42,280] p41196 {2528254882.py:6} INFO - Date range: 2024-10-03 to 2024-12-31


,date,product_id,product_name,category,units_sold,unit_price,total_revenue,region
0,2024-10-03,BOOK003,Mystery Novel Collection,Books,42,24.99,1049.58,Central
1,2024-10-03,ELEC002,USB-C Fast Charger,Electronics,57,24.99,1424.43,Central
2,2024-10-03,BOOK001,Python Programming Guide,Books,39,39.65,1546.35,West


In [5]:
sample_file = sorted(Path(TEXT_DIR).glob("*.txt"))[0]
logger.info(f"Sample product file: {sample_file.name}\n{sample_file.read_text()[:500]}")

[2026-03-05 00:45:42,289] p41196 {2908494583.py:2} INFO - Sample product file: BEAU001_product_page.txt
VITAMIN C SERUM - PRODUCT PAGE

Product: Vitamin C Serum
Brand: GlowLab Skincare
Price: $28.99
SKU: BEAU001
Category: Beauty & Personal Care

PRODUCT DESCRIPTION:
Reveal brighter, more youthful skin with GlowLab's Vitamin C Serum. This powerful
antioxidant formula combines 20% Vitamin C (L-Ascorbic Acid) with Vitamin E and
Hyaluronic Acid to reduce dark spots, fine lines, and sun damage while hydrating
and protecti


## Verify Query Router

In [6]:
expected = ["csv", "csv", "text", "text", "both", "both"]

for q, exp in zip(test_questions, expected):
    logger.info(f"expected: {exp}")
    result = classify_query(q)

[2026-03-05 00:45:42,292] p41196 {1774832203.py:4} INFO - expected: csv
[2026-03-05 00:45:42,293] p41196 {router.py:79} INFO - classify_query: [csv] What was the total revenue for Electronics category in Decem
[2026-03-05 00:45:42,293] p41196 {1774832203.py:4} INFO - expected: csv
[2026-03-05 00:45:42,293] p41196 {router.py:79} INFO - classify_query: [csv] Which region had the highest sales volume?
[2026-03-05 00:45:42,293] p41196 {1774832203.py:4} INFO - expected: text
[2026-03-05 00:45:42,294] p41196 {router.py:79} INFO - classify_query: [text] What are the key features of the Wireless Bluetooth Headphon
[2026-03-05 00:45:42,294] p41196 {1774832203.py:4} INFO - expected: text
[2026-03-05 00:45:42,294] p41196 {router.py:79} INFO - classify_query: [text] What do customers say about the Air Fryer's ease of cleaning
[2026-03-05 00:45:42,294] p41196 {1774832203.py:4} INFO - expected: both
[2026-03-05 00:45:42,294] p41196 {router.py:79} INFO - classify_query: [both] Which product has the b

## Verify CSV Retriever

In [7]:
sample_csv = retrieve_from_csv(
    "Which region had the highest sales volume?",
    csv_path=CSV_PATH,
)
print(sample_csv[:600])

[2026-03-05 00:45:42,298] p41196 {retriever.py:37} INFO - Retrieving from CSV...
[2026-03-05 00:45:42,305] p41196 {retriever.py:132} INFO - CSV context length: 2920 chars


=== Sales Data Overview ===
Total records: 1000
Date range: 2024-10-03 to 2024-12-31
Total revenue (all time): $1,738,661.87
Total units sold (all time): 28,804

=== Revenue by Category ===
category
Electronics               413849.85
Clothing                  352019.03
Office Supplies           185907.72
Home & Kitchen            176182.67
Sports & Outdoors         175596.91
Pet Supplies              120314.34
Beauty & Personal Care     94322.77
Books                      89979.62
Toys & Games               81089.72
Food & Grocery             49399.24

=== Sales Volume (Units) by Region ===
r


## Verify Text Retriever

In [8]:
sample_text = retrieve_from_text(
    "What are the key features of the Wireless Bluetooth Headphones?",
    text_dir=TEXT_DIR,
)
print(sample_text[:600])

[2026-03-05 00:45:42,308] p41196 {retriever.py:152} INFO - Retrieving from text files...
[2026-03-05 00:45:42,309] p41196 {retriever.py:219} INFO - Text context length: 6020 chars from 10 files


=== ELEC001_product_page.txt (relevance score: 105) ===
WIRELESS BLUETOOTH HEADPHONES - PRODUCT PAGE

Product: Wireless Bluetooth Headphones
Brand: SoundMax Pro
Price: $79.99
SKU: ELEC001
Category: Electronics

PRODUCT DESCRIPTION:
Experience crystal-clear audio with our premium Wireless Bluetooth Headphones.
Featuring advanced noise-cancellation technology and 40-hour battery life, these
headphones are perfect for commuting, working from home, or enjoying your favorite
music without interruption.

Key Features:



## Answer Test Questions

In [9]:
all_results = []

for i, question in enumerate(test_questions, 1):
    logger.info("=" * 60)
    logger.info(f"Question [{i}/{len(test_questions)}] {question}")
    result = pipeline.answer(question)
    all_results.append(result)
    logger.info(f"Answer [{i}/{len(test_questions)}]:\n{result['answer']}")
    time.sleep(10)

[2026-03-05 00:45:42,313] p41196 {1674143591.py:4} INFO - ============================================================
[2026-03-05 00:45:42,313] p41196 {1674143591.py:5} INFO - Question [1/6] What was the total revenue for Electronics category in December 2024?
[2026-03-05 00:45:42,314] p41196 {pipeline.py:92} INFO - Question: What was the total revenue for Electronics category in December 2024?
[2026-03-05 00:45:42,314] p41196 {router.py:79} INFO - classify_query: [csv] What was the total revenue for Electronics category in Decem
[2026-03-05 00:45:42,314] p41196 {retriever.py:37} INFO - Retrieving from CSV...
[2026-03-05 00:45:42,321] p41196 {retriever.py:132} INFO - CSV context length: 3332 chars
00:45:42 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-05 00:45:42,329] p41196 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
00:45:42 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed C

## Save Results

In [10]:
output_path = "part2_results.txt"

with open(output_path, "w") as f:
    f.write("Part 2: Multi-Source RAG with Query Routing\n")
    f.write("=" * 60 + "\n\n")
    for i, result in enumerate(all_results, 1):
        f.write(f"Question {i}: {result['question']}\n")
        f.write(f"[{result['query_type']}]\n")
        f.write("Answer:\n")
        f.write(f"{result['answer']}\n\n")
        f.write("=" * 60 + "\n\n")

logger.info(f"Saved to {output_path}")

[2026-03-05 00:46:45,458] p41196 {3756498858.py:13} INFO - Saved to part2_results.txt
